# Day 019 — Exercise 2: LLM-as-Judge

**Goal:** Implement `llm_judge(question, response, expected, model)` that scores a model response on a 1–5 rubric using a structured judge prompt. One real Ollama call is made in the checks.

In [ ]:
import re
import ollama

## Provided: Scoring Metrics

In [ ]:
def exact_match(response: str, expected: str) -> bool:
    return response.strip().lower() == expected.strip().lower()

def contains_any(response: str, keywords: list[str]) -> bool:
    resp_lower = response.lower()
    return any(kw.lower() in resp_lower for kw in keywords)


## Your Implementation

In [ ]:
JUDGE_PROMPT = """\
You are an evaluation judge. Score the response below on a scale of 1 to 5.

Question: {question}
Expected answer: {expected}
Actual response: {response}

Rubric:
1 = Completely wrong or irrelevant
2 = Mostly wrong with minor correct elements
3 = Partially correct but with significant gaps
4 = Mostly correct with minor issues
5 = Fully correct and complete

Respond with ONLY this format:
Score: <1-5>
Rationale: <one sentence>
"""


def llm_judge(
    question: str,
    response: str,
    expected: str,
    model: str = 'llama3.2',
) -> dict:
    """
    Score a model response on a 1-5 rubric using an LLM judge.
    Returns {'score': int, 'rationale': str}.
    Falls back to score=3 and raw text if parsing fails.
    """
    # TODO: format JUDGE_PROMPT with question, expected, response
    # TODO: call ollama.chat with the formatted prompt
    # TODO: extract score with re.search(r'Score:\s*([1-5])', text)
    # TODO: extract rationale with re.search(r'Rationale:\s*(.+)', text)
    # TODO: return {'score': score, 'rationale': rationale}
    pass


## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: llm_judge defined
    try:
        assert 'llm_judge' in globals()
        passed += 1; print('✅ Check 1: llm_judge defined')
    except Exception as e:
        print(f'❌ Check 1: {e}')

    # Check 2: returns a dict (one real Ollama call)
    try:
        result = llm_judge('What is 2 + 2?', '4', '4')
        assert isinstance(result, dict), f'expected dict, got {type(result)}'
        passed += 1; print('✅ Check 2: llm_judge returns a dict')
    except Exception as e:
        print(f'❌ Check 2: return type — {e}')

    # Check 3: score is an int in 1-5
    try:
        assert 'score' in result, f'missing key: score'
        assert isinstance(result['score'], int), f"score must be int, got {type(result['score'])}"
        assert 1 <= result['score'] <= 5, f"score must be 1-5, got {result['score']}"
        passed += 1; print(f"✅ Check 3: score is int in [1,5] — got {result['score']}")
    except Exception as e:
        print(f'❌ Check 3: score — {e}')

    # Check 4: rationale is a non-empty string
    try:
        assert 'rationale' in result, 'missing key: rationale'
        assert isinstance(result['rationale'], str) and len(result['rationale']) > 0
        passed += 1; print('✅ Check 4: rationale is a non-empty string')
    except Exception as e:
        print(f'❌ Check 4: rationale — {e}')

    # Check 5: clearly correct answer scores >= 3
    try:
        r2 = llm_judge('What is the capital of France?', 'Paris', 'Paris')
        assert r2['score'] >= 3, f"expected score >= 3 for perfect answer, got {r2['score']}"
        passed += 1; print(f"✅ Check 5: correct answer scores >= 3 (got {r2['score']})")
    except Exception as e:
        print(f'❌ Check 5: score for correct answer — {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')

_run_checks()


## Solution

<details>
<summary>Click to reveal</summary>

```python
def llm_judge(question, response, expected, model='llama3.2'):
    prompt = JUDGE_PROMPT.format(
        question=question, expected=expected, response=response
    )
    raw = ollama.chat(model=model,
                      messages=[{'role': 'user', 'content': prompt}])
    text = raw['message']['content']
    m = re.search(r'Score:\s*([1-5])', text)
    score = int(m.group(1)) if m else 3
    r = re.search(r'Rationale:\s*(.+)', text)
    rationale = r.group(1).strip() if r else text.strip()[:200]
    return {'score': score, 'rationale': rationale}
```

</details>